# 🪄 Weights & Biases (WandB) Demo: MLOps for Everyone

This notebook provides a **hands-on, beginner-friendly** demonstration of **Weights & Biases (WandB)**, a powerful platform for experiment tracking, dataset versioning, and model management.

## 📚 What You'll Learn

By the end of this notebook, you will be able to:

1. ✅ **Setup WandB** - Create an account and link it to your notebook
2. ✅ **Track Experiments** - Log hyperparameters (config) and metrics (accuracy, loss)
3. ✅ **Visualize Results** - Create stunning interactive plots automatically
4. ✅ **Compare Runs** - Analyze hyperparameter sweeps in the dashboard
5. ✅ **Version Models** - Use WandB Artifacts to save and track model versions
6. ✅ **Load Models** - Retrieve specific model versions for inference

## 🎯 Why WandB?

WandB is popular in the Deep Learning community because it's:
- 🎨 **Visual**: Beautiful, interactive dashboards out of the box
- ☁️ **Hosted**: No need to set up your own server (unlike local MLflow)
- 🤝 **Collaborative**: Share results with a single link
- 🔌 **Integrations**: Works seamlessly with PyTorch, TensorFlow, Scikit-learn, Hugging Face, etc.

---


## 1. Setup & Login

First, we need to install the library and log in.

**🛑 Prerequisite:**
1. Go to [wandb.ai](https://wandb.ai/site) and sign up for a free account.
2. Go to your [Settings/API Keys](https://wandb.ai/settings) page to copy your API Key.


In [1]:
!uv pip install wandb scikit-learn matplotlib

Using Python 3.10.18 environment at: /Users/tarekatwan/Repos/MyWork/Teach/repos/advanced_machine_learning/.venv
Audited 3 packages in 37ms


In [2]:
import wandb
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_iris
import pandas as pd
import numpy as np
import os

# Login to WandB
# You will be asked to paste your API key here the first time
wandb.login()

wandb: Currently logged in as: tatwan to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 🧠 Core Concepts

- **Project**: A collection of runs (e.g., "Iris-Classification").
- **Run**: A single execution of your training script.
- **Config**: Hyperparameters (inputs) like `learning_rate`, `batch_size`.
- **Summary**: Final metrics (outputs) like `best_accuracy`.
- **History**: Metrics tracked over time (e.g., loss per epoch).
- **Artifacts**: Files (datasets, models) that are versioned and tracked.

---


## 2. Basic Experiment Tracking

Let's train a simple Logistic Regression model and track it with WandB.

### Key Steps:
1. `wandb.init()`: Start a new run.
2. `wandb.config`: Save your hyperparameters.
3. `wandb.log()`: Log metrics and charts.
4. `wandb.finish()`: End the run (important in notebooks!).


In [3]:
# 1. Load Data
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Initialize WandB Run
run = wandb.init(
    project="wandb-demo-iris",
    name="Logistic_Regression_Baseline",
    tags=["baseline", "linear"],
    config={
        "model": "LogisticRegression",
        "solver": "lbfgs",
        "max_iter": 1000,
        "random_state": 42
    }
)

# 3. Train Model
model = LogisticRegression(
    solver=wandb.config.solver,
    max_iter=wandb.config.max_iter,
    random_state=wandb.config.random_state
)
model.fit(X_train, y_train)

# 4. Evaluate
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

# 5. Log Metrics
wandb.log({"accuracy": accuracy})
print(f"✅ Run finished. Accuracy: {accuracy}")

# 6. Finish Run
wandb.finish()

✅ Run finished. Accuracy: 1.0


accuracy,▁
accuracy,1


## 3. Rich Visualizations 🎨

WandB shines with its built-in plotting capabilities. For Scikit-Learn, it can automatically generate Confusion Matrices, ROC Curves, and more!


In [4]:
# Start a new run for visualization
wandb.init(project="wandb-demo-iris", name="Visualizations_Demo", config={"model": "LogisticRegression"})

# Train model again
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_probas = model.predict_proba(X_test)

# Log Scikit-learn plots
# These create interactive charts in your dashboard!
wandb.sklearn.plot_classifier(
    model, 
    X_train, X_test, 
    y_train, y_test, 
    y_pred, y_probas, 
    iris.target_names, 
    feature_names=iris.feature_names
)

print("✅ Visualizations logged! Check your WandB dashboard link above.")
wandb.finish()

wandb: 
wandb: Plotting Classifier.
wandb: WARNING 2-dimensional feature importances array passed to plot_feature_importances. 2-dimensional and higher feature importances arrays are not currently supported. These importances will not be plotted.
wandb: Logged feature importances.
wandb: Logged confusion matrix.
wandb: Logged summary metrics.
wandb: Logged class proportions.
wandb: WARNING This function only supports binary classification at the moment and therefore expects labels to be binary. Skipping calibration curve.
wandb: Logged calibration curve.
wandb: Logged roc curve.
wandb: Logged precision-recall curve.


✅ Visualizations logged! Check your WandB dashboard link above.


## 4. Comparing Runs (Hyperparameter Tuning)

Just like in our MLflow demo, let's run a loop to try different hyperparameters and see how they compare in the WandB dashboard.


In [5]:
from sklearn.ensemble import RandomForestClassifier

# Define hyperparameter grid
n_estimators_list = [10, 50, 100]
max_depth_list = [3, 5, None]

print("🚀 Starting hyperparameter sweep...")

for n_est in n_estimators_list:
    for depth in max_depth_list:
        # Start a run for each combination
        with wandb.init(
            project="wandb-demo-iris",
            name=f"RF_n{n_est}_d{depth}",
            config={
                "model": "RandomForest",
                "n_estimators": n_est,
                "max_depth": depth
            },
            reinit=True # Allow multiple runs in same script
        ) as run:
            
            # Train
            rf = RandomForestClassifier(n_estimators=n_est, max_depth=depth, random_state=42)
            rf.fit(X_train, y_train)
            
            # Evaluate
            acc = rf.score(X_test, y_test)
            
            # Log metric
            from sklearn.metrics import f1_score, precision_score

            y_pred = rf.predict(X_test)
            f1 = f1_score(y_test, y_pred, average='weighted')
            precision = precision_score(y_test, y_pred, average='weighted')

            wandb.log({"accuracy": acc, "f1_score": f1, "precision_score": precision})
            
            print(f"Logged: n_estimators={n_est}, max_depth={depth} -> Acc={acc:.4f}")

print("✅ Sweep complete! Go to your dashboard to compare runs using the Parallel Coordinates Plot.")

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


🚀 Starting hyperparameter sweep...


Logged: n_estimators=10, max_depth=3 -> Acc=1.0000


accuracy,▁
f1_score,▁
precision_score,▁
accuracy,1
f1_score,1
precision_score,1


Logged: n_estimators=10, max_depth=5 -> Acc=1.0000


...

Logged: n_estimators=100, max_depth=5 -> Acc=1.0000


accuracy,▁
f1_score,▁
precision_score,▁
accuracy,1
f1_score,1
precision_score,1


Logged: n_estimators=100, max_depth=None -> Acc=1.0000


accuracy,▁
f1_score,▁
precision_score,▁
accuracy,1
f1_score,1
precision_score,1


✅ Sweep complete! Go to your dashboard to compare runs using the Parallel Coordinates Plot.


## 5. Model Versioning with Artifacts 📦

WandB Artifacts allow you to save datasets and models, version them, and track their lineage.

**Workflow:**
1. Save model to a local file.
2. Create an Artifact object.
3. Add the file to the Artifact.
4. Log the Artifact.


In [6]:
import joblib

# Start a run to save the best model
with wandb.init(project="wandb-demo-iris", name="Model_Registry_Demo") as run:
    
    # Train a model
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    # 1. Save locally
    model_path = "model.joblib"
    joblib.dump(model, model_path)
    
    # 2. Create Artifact
    artifact = wandb.Artifact(
        name="iris-random-forest", 
        type="model",
        description="Random Forest model with 100 estimators"
    )
    
    # 3. Add file
    artifact.add_file(model_path)
    
    # 4. Log Artifact
    run.log_artifact(artifact)
    
    print("✅ Model artifact logged!")

# Cleanup local file
if os.path.exists(model_path):
    os.remove(model_path)

✅ Model artifact logged!


## 6. Loading Models 🔄

To use a model in production or for further testing, you can download it from WandB.


In [7]:
# Start a consumer run
with wandb.init(project="wandb-demo-iris", name="Model_Loading_Demo") as run:
    
    # 1. Use the artifact (format: entity/project/name:version)
    # 'latest' is a handy alias for the most recent version
    artifact = run.use_artifact('iris-random-forest:latest')
    
    # 2. Download the directory
    artifact_dir = artifact.download()
    
    # 3. Load the model
    model_path = os.path.join(artifact_dir, "model.joblib")
    loaded_model = joblib.load(model_path)
    
    # 4. Make predictions
    sample = [X_test[0]]
    pred = loaded_model.predict(sample)
    print(f"🔮 Prediction for {sample}: {iris.target_names[pred][0]}")
    
    print(f"✅ Model loaded from: {artifact_dir}")

wandb:   1 of 1 files downloaded.  


🔮 Prediction for [array([6.1, 2.8, 4.7, 1.2])]: versicolor
✅ Model loaded from: /Users/tarekatwan/Repos/MyWork/Teach/repos/advanced_machine_learning/activities/mlops/artifacts/iris-random-forest:v0


## Conclusion & Best Practices 🌟

You've now mastered the basics of WandB!

### 💡 Best Practices:
1. **Config is Key**: Always log your hyperparameters in `wandb.config`. It enables the powerful comparison tools.
2. **Tags**: Use tags like `["production"]`, `["experiment"]` to organize runs.
3. **Artifacts**: Use Artifacts for ANY file you want to version (datasets, models, preprocessing pipelines).
4. **Group**: Use `group` parameter in `wandb.init` to group related runs (e.g., cross-validation folds).

### 📚 Resources
- [WandB Documentation](https://docs.wandb.ai/)
- [WandB Scikit-Learn Integration](https://docs.wandb.ai/guides/integrations/scikit)
